# Match Cena ↔ Versículo

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

Sugere, pra cada versículo do capítulo, qual vídeo/imagem da sua biblioteca
(`pixabay_stock` ou `image-stock`) melhor combina — usando a sobreposição
entre as tags da lista fechada de Bíblia (`Tags_Biblia_PT`) já preenchidas
na planilha, e tags sugeridas por IA pra cada versículo especificamente.

**Não escreve nada automaticamente** — só gera uma lista pra você revisar.
Versículos sem nenhum vídeo com tag em comum aparecem marcados como
"sem opção", já com palavras-chave sugeridas prontas pra você usar na busca
ao vivo do Pixabay (mesmo diálogo que você já usa).

Funciona igual pra vídeo ou imagem — só muda `TIPO_FONTE` na Configuração.


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q -U groq gspread "mistralai>=1.2.0"

import shutil, sys, json
from pathlib import Path

from google.colab import drive, auth, userdata
from google.auth import default
import gspread
from groq import Groq
from mistralai.client import Mistral

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (versionado; chega pela cópia)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Ele é versionado — rode o repositorio-sincronizar.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── O ambiente combina com o que este notebook faz? ────────────────────────
# Cota de GPU do Colab é limitada e some sem aviso -- e parte da nossa foi
# gasta em notebook que não usa GPU pra nada, rodando com GPU só porque a
# seleção ficou de antes. Silencioso quando combina.
try:
    from ambiente import avisar_gpu
    avisar_gpu(precisa=False)
except Exception:
    pass

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

GROQ_API_KEY = userdata.get("GROQ_KEY")
MISTRAL_API_KEY = userdata.get("MISTRAL_KEY")
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
mistral_client = Mistral(api_key=MISTRAL_API_KEY) if MISTRAL_API_KEY else None

print("✅ Setup concluído")
print(f"   Groq:    {'disponível' if groq_client else '❌ GROQ_KEY não encontrada'}")
print(f"   Mistral: {'disponível' if mistral_client else '❌ MISTRAL_KEY não encontrada'}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.9 MB/s eta 0:00:00
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
✅ 21 módulos copiados
✅ Setup concluído
   Groq:    disponível
   Mistral: disponível


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Vídeo/capítulo (mesmo padrão dos outros notebooks do projeto) ──────────
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"
IDIOMA_MESTRE = "en"
NOME_LEGENDA_MESTRE = "40_Matt_02_whisper_en.srt"
CAPITULO = 2

# Nome do livro em português EXATAMENTE como aparece em titulos-biblicos.js/
# eventos-biblicos.js (ex: "Mateus", "1 Coríntios", "Apocalipse") -- usado
# pra pré-seleção via léxico, ver seção 3b abaixo.
LIVRO_PT = "Mateus"

# Cole aqui o texto do capítulo com os números de versículo isolados no
# meio do fluxo (mesmo formato usado no indicador "Matt 2:4"):
TEXTO_VERSICULOS = """"""

# ── Biblioteca a usar: "video" ou "imagem" ──────────────────────────────────
TIPO_FONTE = "video"

ID_PLANILHA_VIDEOS = "1bF7hnGSY7AALm4ZAS5owWNpiSTdgArW4ahAuVZaHPL0"
NOME_ABA_VIDEOS = "pixabay_stock"

ID_PLANILHA_IMAGENS = "1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E"
NOME_ABA_IMAGENS = "image-stock"

# ── 3b. LÉXICO BÍBLICO (pré-seleção sem IA — ver célula 3b) ────────────────
# Usa titulos-biblicos.js/eventos-biblicos.js (convertidos pra JSON) do
# projeto Glossário como primeira tentativa de tag, ANTES de chamar IA --
# grátis, instantâneo, e mais preciso que a IA pra versículos com cobertura
# no léxico. IA só entra pros versículos sem cobertura (ou como contexto
# extra no prompt, ver match_pipeline.py). Desligue se quiser voltar ao
# comportamento 100% IA de antes.
USAR_LEXICO_BIBLICO = True
NOME_ARQUIVO_TITULOS = "titulos-biblicos.json"
NOME_ARQUIVO_EVENTOS = "eventos-biblicos.json"
# Pasta no Drive onde estão os 2 arquivos acima (fora da pasta do vídeo --
# é um dado compartilhado do projeto Glossário, não deste vídeo específico)
PASTA_DADOS_LEXICO = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/dados_lexico"

# ── 3c. BIBLIOTECA DE MATCH (cache reutilizável entre vídeos) ──────────────
# Cada título/evento que já foi resolvido (léxico ou IA) fica salvo aqui
# (aba nova dentro da MESMA planilha de vídeos/imagens) -- da próxima vez
# que QUALQUER vídeo passar por esse mesmo título/evento, usa direto, sem
# gastar léxico nem IA de novo. Cria a aba sozinho na primeira vez.
USAR_BIBLIOTECA_MATCH = True
# Planilha independente (não fica mais dentro da planilha de vídeos).
# Deixe em branco na primeira vez -- o notebook cria a planilha sozinho
# e IMPRIME o ID aqui embaixo; copie esse ID pra cá pra reusar nas
# próximas vezes (senão cria uma planilha nova toda vez!).
ID_PLANILHA_BIBLIOTECA_MATCH = "1i67VxksAkWYx1cZ_QeoesGXsW28hcA0p5IIfhjx8VHE"
NOME_ABA_BIBLIOTECA_MATCH = "biblioteca_match"

# ── Tags por versículo (pra ranquear candidatos no painel de revisão) ──────
# Mesma planilha da biblioteca_match. Grava as palavras_chave calculadas
# acima (léxico/IA) -- não gasta IA de novo, é só um efeito colateral do
# match. Requer USAR_BIBLIOTECA_MATCH=True (usa a mesma planilha/conexão).
USAR_VERSICULO_TAGS = True
NOME_ABA_VERSICULO_TAGS = "versiculo_tags"

# ── Modelos de IA (texto só — bem mais barato que a descrição de cena,     ──
# ── que usa imagem) ─────────────────────────────────────────────────────────
MODELO_GROQ = "qwen/qwen3.6-27b"
MODELO_MISTRAL = "mistral-small-latest"
DELAY_SEGUNDOS = 2
MAX_TOKENS_RESPOSTA = 300

# ── Anti-repetição ──────────────────────────────────────────────────────────
DIST_MIN_REPETICAO = 3    # nao repete o mesmo video/imagem em versiculos a menos de N de distancia...
MARGEM_PARA_REPETIR = 2   # ...a nao ser que a alternativa mais proxima perca por essa margem de score

print("=" * 60)
print("⚙️  CONFIGURAÇÃO")
print("=" * 60)
print(f"   Vídeo:        {NOME_ORACAO}")
print(f"   Capítulo:     {LIVRO_PT} {CAPITULO}")
print(f"   Fonte:        {TIPO_FONTE}")
print(f"   Léxico:       {'ON' if USAR_LEXICO_BIBLICO else 'off (100% IA)'}")
print(f"   Biblioteca:   {'ON' if USAR_BIBLIOTECA_MATCH else 'off'}")
print(f"   Tags/versic.: {'ON' if USAR_VERSICULO_TAGS else 'off'}")
print(f"   Versículos:   {'(vazio! preencha TEXTO_VERSICULOS)' if not TEXTO_VERSICULOS.strip() else 'preenchido'}")
print("=" * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. INICIALIZAR — carrega a legenda mestre + abre a biblioteca   ║
# ╚══════════════════════════════════════════════════════════════════╝
from config import PipelineConfig
from srt_utils import ler_srt, texto_por_versiculo
from match_pipeline import carregar_biblioteca, gerar_sugestoes_match, carregar_lexico_biblico, abrir_ou_criar_biblioteca_match, carregar_biblioteca_match, garantir_aba_versiculo_tags, carregar_versiculo_tags, garantir_aba_evento_tags, garantir_aba_titulo_tags, carregar_evento_tags, carregar_titulo_tags

config = PipelineConfig(
    NOME_ORACAO=NOME_ORACAO,
    PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ,
    IDIOMA_MESTRE=IDIOMA_MESTRE,
    NOME_LEGENDA_MESTRE=NOME_LEGENDA_MESTRE,
)

# Baixa a legenda mestre direto do Drive (mesma pasta dos outros notebooks)
import shutil as _shutil
from drive_utils import DriveClient
_drive = DriveClient.get()
_destino_mestre = Path(config.nome_legenda_mestre)
_drive.download(config.pasta_oracao, config.nome_legenda_mestre, _destino_mestre)
legendas_mestre = ler_srt(_destino_mestre)
print(f"✅ Legenda mestre carregada: {len(legendas_mestre)} blocos")

# Extrai o texto de cada versículo (sem timing — só o conteúdo).
# Tenta baixar o arquivo padronizado primeiro (videos/<nome>/<nome>_roteiro_versiculos.txt
# -- gerado pelo limpar_roteiro_biblia.html); se não achar, cai pro texto
# colado na célula de Configuração.
if not TEXTO_VERSICULOS.strip():
    _dest_roteiro_versiculos = Path(config.nome_palavras_mestre)
    _drive.download_se_ausente(config.pasta_oracao, config.nome_palavras_mestre, _dest_roteiro_versiculos)
    if _dest_roteiro_versiculos.exists():
        TEXTO_VERSICULOS = _dest_roteiro_versiculos.read_text(encoding="utf-8")
        print(f"✅ TEXTO_VERSICULOS carregado do arquivo ({config.nome_palavras_mestre})")
    else:
        raise ValueError(
            f"TEXTO_VERSICULOS está vazio e {config.nome_palavras_mestre} não foi encontrado "
            f"(local nem no Drive) -- cole o texto na célula de Configuração, ou gere o arquivo "
            f"com limpar_roteiro_biblia.html e suba pra videos/{NOME_ORACAO}/."
        )
versiculos_texto = texto_por_versiculo(TEXTO_VERSICULOS)
print(f"✅ {len(versiculos_texto)} versículos extraídos")

# Abre a planilha certa conforme TIPO_FONTE
if TIPO_FONTE == "video":
    id_planilha, nome_aba, coluna_url, coluna_tags_biblia = ID_PLANILHA_VIDEOS, NOME_ABA_VIDEOS, "url", "Tags_Biblia_PT"
elif TIPO_FONTE == "imagem":
    if not ID_PLANILHA_IMAGENS:
        raise ValueError("TIPO_FONTE='imagem' mas ID_PLANILHA_IMAGENS não foi preenchido.")
    id_planilha, nome_aba, coluna_url, coluna_tags_biblia = ID_PLANILHA_IMAGENS, NOME_ABA_IMAGENS, "Imagem", "Tags_Semelhantes_PT"
else:
    raise ValueError(f"TIPO_FONTE inválido: {TIPO_FONTE!r} (use 'video' ou 'imagem')")

sheet = gc.open_by_key(id_planilha).worksheet(nome_aba)
linhas_planilha = sheet.get_all_records()
biblioteca = carregar_biblioteca(linhas_planilha, coluna_url=coluna_url, coluna_tags_biblia=coluna_tags_biblia)
print(f"✅ Biblioteca ({TIPO_FONTE}): {len(linhas_planilha)} linhas na planilha, "
      f"{len(biblioteca)} já têm {coluna_tags_biblia} preenchida (só essas entram no match)")
# ── 3b. Carrega o léxico bíblico (opcional, ver USAR_LEXICO_BIBLICO) ────────
titulos_biblicos = eventos_biblicos = None
if USAR_LEXICO_BIBLICO:
    _dest_titulos = Path(NOME_ARQUIVO_TITULOS)
    _dest_eventos = Path(NOME_ARQUIVO_EVENTOS)
    _drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_TITULOS, _dest_titulos)
    _drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_EVENTOS, _dest_eventos)
    if _dest_titulos.exists() and _dest_eventos.exists():
        titulos_biblicos, eventos_biblicos = carregar_lexico_biblico(_dest_titulos, _dest_eventos)
        print(f"✅ Léxico bíblico: {len(titulos_biblicos)} títulos, {len(eventos_biblicos)} eventos")
    else:
        print(f"⚠️  Léxico não encontrado em {PASTA_DADOS_LEXICO} (local nem Drive) -- caindo pra 100% IA")

# ── 3c. Abre/cria a aba de biblioteca de match + carrega o que já existe ────
biblioteca_match = {}
aba_biblioteca_match = None
if USAR_BIBLIOTECA_MATCH:
    _spreadsheet_biblioteca, aba_biblioteca_match, _id_usado = abrir_ou_criar_biblioteca_match(gc, ID_PLANILHA_BIBLIOTECA_MATCH, NOME_ABA_BIBLIOTECA_MATCH)
    biblioteca_match = carregar_biblioteca_match(aba_biblioteca_match)
    print(f"✅ Biblioteca de match: {len(biblioteca_match)} título(s)/evento(s) já resolvido(s) antes")

# ── 3d. Abre/cria a aba de tags por versículo ───────────────────────────────
aba_versiculo_tags = None
if USAR_VERSICULO_TAGS:
    if not USAR_BIBLIOTECA_MATCH:
        print("⚠️  USAR_VERSICULO_TAGS precisa de USAR_BIBLIOTECA_MATCH=True (usa a mesma planilha) -- ignorando.")
    else:
        aba_versiculo_tags = garantir_aba_versiculo_tags(_spreadsheet_biblioteca, NOME_ABA_VERSICULO_TAGS)
        print(f"✅ Tags por versículo: aba pronta ({len(carregar_versiculo_tags(aba_versiculo_tags))} versículo(s) já taggeado(s))")

# ── 3e. Abre/cria e carrega evento_tags/titulo_tags (fonte editável na planilha) ──
aba_evento_tags = garantir_aba_evento_tags(_spreadsheet_biblioteca)
aba_titulo_tags = garantir_aba_titulo_tags(_spreadsheet_biblioteca)
evento_tags_dict = carregar_evento_tags(aba_evento_tags)
titulo_tags_dict = carregar_titulo_tags(aba_titulo_tags)
print(f"✅ evento_tags/titulo_tags carregados da planilha ({len(evento_tags_dict)} evento(s), {len(titulo_tags_dict)} título(s))")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. LISTA FECHADA DE TAGS_BIBLIA (mesma do notebook de descrição) ║
# ╚══════════════════════════════════════════════════════════════════╝
LISTA_TAGS_BIBLIA = ("criação, jardim do éden, queda, dilúvio, arca de noé, torre de babel, "
"chamado de abraão, aliança, sacrifício de isaque, jacó e esaú, escada de jacó, josé e os "
"irmãos, sonhos proféticos, escravidão no egito, sarça ardente, pragas do egito, travessia do "
"mar vermelho, maná no deserto, dez mandamentos, monte sinai, bezerro de ouro, tabernáculo, "
"arca da aliança, peregrinação no deserto, serpente de bronze, terra prometida, queda de "
"jericó, juízes, sansão, gideão, débora, rute e noemi, davi e golias, davi e saul, reino de "
"davi, sabedoria de salomão, templo de salomão, reino dividido, elias no monte carmelo, carro "
"de fogo, eliseu, exílio babilônico, daniel na cova dos leões, fornalha ardente, jonas e o "
"grande peixe, ester, sofrimento de jó, salmos e louvor, provérbios e sabedoria, reconstrução "
"do templo, profecia messiânica, anunciação, natividade, magos do oriente, estrela de belém, "
"fuga para o egito, apresentação no templo, batismo de jesus, tentação no deserto, chamado dos "
"discípulos, sermão da montanha, bem-aventuranças, milagre de cura, multiplicação dos pães, "
"tempestade acalmada, jesus anda sobre as águas, parábola do semeador, parábola do filho "
"pródigo, parábola do bom samaritano, ovelha perdida, transfiguração, ressurreição de lázaro, "
"entrada triunfal em jerusalém, última ceia, getsêmani, prisão e julgamento, crucificação, "
"ressurreição de jesus, tumba vazia, estrada de emaús, ascensão, pentecostes, conversão de "
"paulo, viagens missionárias, igreja primitiva, perseguição dos cristãos, cartas apostólicas, "
"apocalipse, pastor e ovelhas, boas novas, anjo mensageiro, profeta, rei, sacerdote, juízo, "
"misericórdia divina, aliança renovada, êxodo espiritual, batalha espiritual, jornada de fé, "
"provação, milagre, cura, ressurreição, segunda vinda, reino de deus, cordeiro de deus, luz "
"do mundo")

print(f"{len(LISTA_TAGS_BIBLIA.split(','))} temas na lista fechada")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. RODAR O MATCH                                                ║
# ╚══════════════════════════════════════════════════════════════════╝
if not (groq_client or mistral_client):
    print("⚠️  Nenhuma API de IA disponível (GROQ_KEY/MISTRAL_KEY) -- só versículos já cobertos "
          "pela biblioteca de match ou pelo léxico vão resolver; o resto fica 'sem opção'.")

resultados = gerar_sugestoes_match(
    versiculos_texto, biblioteca, LISTA_TAGS_BIBLIA,
    groq_client, mistral_client, MODELO_GROQ, MODELO_MISTRAL,
    dist_min_repeticao=DIST_MIN_REPETICAO, margem_para_repetir=MARGEM_PARA_REPETIR,
    delay_segundos=DELAY_SEGUNDOS, max_tokens=MAX_TOKENS_RESPOSTA,
    livro_pt=LIVRO_PT if USAR_LEXICO_BIBLICO else None, capitulo=CAPITULO,
    titulos_biblicos=titulos_biblicos, eventos_biblicos=eventos_biblicos,
    tipo_fonte=TIPO_FONTE, biblioteca_match=biblioteca_match, aba_biblioteca_match=aba_biblioteca_match,
    aba_versiculo_tags=aba_versiculo_tags,
)

com_match = sum(1 for r in resultados if not r["sem_opcao"])
print(f"\n{'='*60}")
print(f"✅ {len(resultados)} versículos processados")
print(f"   Com sugestão:  {com_match}")
print(f"   Sem opção:     {len(resultados) - com_match}")
print(f"{'='*60}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. REVISAR OS RESULTADOS                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
for r in resultados:
    cap_v = f"{CAPITULO}:{r['versiculo']}"
    if r["sem_opcao"]:
        print(f"❌ {cap_v:6s} SEM OPÇÃO — busque por: {', '.join(r['palavras_chave']) or '(nenhuma sugestão)'}")
    else:
        emoji_fonte = {"biblioteca": "🗃️", "lexico": "📚"}.get(r.get("fonte"), "🤖")  # 🗃️ biblioteca / 📚 lexico / 🤖 IA
        print(f"✅ {cap_v:6s} {emoji_fonte} [{r['id']}] {r['titulo']}  (score={r['score']}, bateu em: {', '.join(r['tags_batidas'])})")

# Salva os resultados num JSON, pra usar depois (ex: alimentar o baixar_clipes())
nome_arquivo = config.nome_match_json(CAPITULO)
with open(nome_arquivo, "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print(f"\n💾 Resultados salvos em {nome_arquivo}")

# Sobe pro Drive também (mesma pasta do vídeo) -- sem isso, o próximo notebook
# (video-base-*-versiculo.ipynb) não acha o arquivo, porque cada notebook Colab
# roda numa sessão/runtime separada (o download local daqui não chega lá sozinho).
_drive.upload(Path(nome_arquivo), config.pasta_oracao, "application/json")
print(f"☁️  Também enviado pro Drive: {config.pasta_oracao / nome_arquivo}")

from google.colab import files
files.download(nome_arquivo)
